# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'
dataset = mlc.Dataset(croissant_url)

# Access the metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The FAIR² Croissant schema describes the record sets as entities referenced by their `@id`. All further exploration and data extraction operate via these `@id`s, as per Croissant and FAIR best practices.

In [ ]:
# Inspect available record sets and their fields
record_sets = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"- {rs['@id']} | {rs.get('schema:name', rs.get('name', ''))}")
    print("  Fields:")
    for field in rs['fields']:
        print(f"    - {field['@id']} | {field.get('schema:name', field.get('name', ''))}")
    print("  Columns:")
    for col in rs.get('columns', []):
        print(f"    - {col['@id']} | {col.get('schema:name', col.get('name', ''))}")
    print()

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis.

We will use the `@id` values found above to extract the record sets, referencing only their `@id`.

All loaded DataFrames will be keyed by their record set `@id`.

In [ ]:
# Get all record set @id values
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    # Load records for this record set using the @id
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df

# Print columns for each record set loaded
for rs_id, df in dataframes.items():
    print(f"\nRecord Set: {rs_id}")
    print(f"Columns: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps on one of the main record sets.

For demonstration, we'll:
- Select a numeric field (e.g., GAD-7 score or PHQ-9 score) by its `@id`
- Filter records based on a threshold (e.g., mental health score > 10),
- Normalize the numeric field,
- Group the data by a key attribute (e.g., gender/age).

In [ ]:
# For illustration, select the main survey record set (replace with actual @id from previous output as needed)
# Example: main_record_set_id = 'https://api.app.sen.science/frontiers/7160411/mental-health-records'
if len(dataframes):
    main_record_set_id = list(dataframes.keys())[0]  # Use the first available record set
    df = dataframes[main_record_set_id]
    # Identify a numeric field -- insert the actual @id if available (e.g., 'GAD-7-score', 'PHQ-9-score')
    # For demonstration, use a column ending in 'score'. Adjust as needed based on actual column names.
    numeric_field = None
    for col in df.columns:
        if 'score' in col.lower():
            numeric_field = col
            break
    if numeric_field:
        print(f"Using numeric field for analysis: {numeric_field}")
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field}:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a key field (e.g., gender or age group). Use actual @id if available.
        possible_group_fields = [col for col in df.columns if ('gender' in col.lower() or 'age' in col.lower())]
        if possible_group_fields:
            group_field = possible_group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable grouping field (e.g., gender/age) found.")
    else:
        print("No numeric score field found in the main record set.")

## 5. Visualization
Visualize distributions and relationships between fields in the dataset.

Let's plot (if available) the distribution of mental health scores for the selected record set and grouping attribute.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes):
    df = dataframes[main_record_set_id]
    if numeric_field:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Frequency")
        plt.show()

        if possible_group_fields:
            group_field = possible_group_fields[0]
            plt.figure(figsize=(8, 4))
            sns.boxplot(x=group_field, y=numeric_field, data=df)
            plt.title(f"{numeric_field} by {group_field}")
            plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² mental health survey dataset is accessible via Croissant schema and the `mlcroissant` library.
- All entities—including record sets and fields—are referenced strictly via their `@id`, supporting interoperability and reproducible science.
- We extracted and reviewed demographic and mental health score data with common EDA operations: filtering, normalization, and grouping.
- Visualizations highlight the distribution and variation of mental health scores across demographic groups.

Further analyses can expand to explore survey responses in depth and integrate additional fields and record sets.